# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring

**Task type:** Ranking / scoring, supported by classification.

The main task is to rank content pages by how strongly they should be prioritised for human review. A classification model can estimate the probability that a page belongs to a defined risk or opportunity group, and this probability can be used as a score to rank pages.

The final output is therefore not only a yes/no prediction. It is a ranked review queue that helps an SEO or content team decide which pages to inspect first.

The output supports actions such as refreshing, expanding, protecting, pruning, or monitoring a page after human review.

## 2. Target or proxy

For the final version of this ranking problem, I want the target to come from an observed outcome rather than from a hand-written decision rule.

A stronger target would use an earlier feature window and a later outcome window, for example:

prior 90 days of observable signals → decline or recovery during the next 30 days

This would allow the ranking score to represent future observed risk or opportunity.

For the starter dataset, I can temporarily use `trend_direction == "down"` as a teaching proxy. However, this is not my ideal capstone target because `trend_direction` describes the current measurement window rather than a separately observed future outcome.

I will therefore treat the starter label as a proxy for learning and initial comparison, not as the final definition of which pages truly deserve a refresh."

In [8]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["starter_proxy"] = (
    df["trend_direction"]
      .str.lower()
      .eq("down")
      .astype(int)
)

df[[
    "content_id",
    "trend_direction",
    "starter_proxy"
]].head(10)

,content_id,trend_direction,starter_proxy
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


## 3. Success metric

**Primary metric: Precision@50**

Precision@50 measures the proportion of the top 50 pages in my ranked review queue that are positive according to the defined target.

I chose this metric because the real task is prioritisation. If a content team can review only a limited number of pages, the quality of the pages near the top of the ranking matters more than overall classification accuracy.

A higher Precision@50 means that more of the team's limited review capacity is being directed toward pages that match the defined risk or opportunity target.

I will compare the ML ranking against a transparent rule-based baseline rather than choosing an arbitrary value that automatically counts as "good".

In [5]:
def precision_at_k(scores, labels, k=50):
    order = scores.argsort()[::-1]
    return labels.iloc[order[:k]].mean()

print("Primary evaluation metric: Precision@50")
print("Review capacity represented by K:", 50)

Primary evaluation metric: Precision@50
Review capacity represented by K: 50


## 4. The unit of analysis, as a real dataframe

**Unit of analysis: one pseudonymised content page.**

Each row in the starter dataset represents one content item/page with observable search, content, freshness, and engagement signals.

For this lane, I use these page-level signals to decide how strongly each page should be prioritised for review.

In [6]:
lane_columns = [
    "content_id",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "engagement_rate",
    "trend_direction",
    "is_declining_label"
]

lane_df = df[lane_columns].copy()

print("Shape:", lane_df.shape)
print("One row = one pseudonymised content page")

lane_df.head(10)

Shape: (30000, 10)
One row = one pseudonymised content page


,content_id,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,engagement_rate,trend_direction,is_declining_label
0,content_304f48230142,3803,10.6,0.76,187,20,3221.0,5.88,down,1
1,content_a1fb4e703a9e,15320,20.3,0.05,445,25,2481.0,0.00,down,1
2,content_9aa793d4d895,12581,36.5,0.09,141,20,3515.0,0.00,down,1
3,content_331d6c4de07b,11751,6.2,0.49,463,22,NaN,1.28,stable,0
4,content_d99b7a2d90ca,19140,44.0,0.13,263,14,2803.0,0.00,down,1
5,content_d4084a4bc775,3970,8.5,0.03,147,20,3080.0,0.00,down,1
6,content_9a34b442b552,20,7.0,0.00,90,20,3059.0,0.00,down,1
7,content_a63219c6e95a,1724,21.2,0.06,445,22,NaN,3.57,stable,0
8,content_5e6c160719bc,32574,46.0,0.09,90,20,3807.0,5.88,down,1
9,content_c27558df2b0c,1240,4.9,0.16,257,104,NaN,0.00,down,1


## 5. Why ML may beat a fixed rule here

A fixed rule such as "review pages that are at least 180 days stale and have at least 500 impressions" is simple, transparent, and useful as a baseline.

However, refresh priority may depend on several interacting signals. For example, two pages with the same number of impressions may have different positions, CTR, freshness, age, and engagement.

A fixed rule requires me to manually choose thresholds and interactions. A machine-learning model can learn useful combinations of these observable signals from the data.

ML is not automatically better. I will compare it against the transparent baseline using Precision@50 and honest validation. The additional complexity is worthwhile only if it produces a meaningfully better review ranking.

The final system should still provide understandable reason codes so that a human reviewer can understand why a page was prioritised.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.